# 🤖 AI-Based Autonomous Navigation System — Interactive Demo

This notebook walks through the full pipeline interactively.

**Run all cells top-to-bottom.**

In [ ]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
%matplotlib inline
print('Ready ✓')

## 1. Build the Environment

In [ ]:
from simulation.environment import Environment

env = Environment(rows=20, cols=20, obstacle_ratio=0.18, n_dynamic=4, seed=42)

colors = ['#F8F9FA','#343A40','#F77F00','#0077B6','#2DC653','#ADE8F4']
cmap   = mcolors.ListedColormap(colors)

fig, ax = plt.subplots(figsize=(7,7))
ax.imshow(env.render(), cmap=cmap, vmin=0, vmax=5, interpolation='nearest')
ax.set_title('Environment Map', fontsize=14)
plt.tight_layout(); plt.show()

print(f'Grid: {env.rows}×{env.cols}')
print(f'Start: {env.agent_pos}   Goal: {env.goal_pos}')

## 2. Path Planning — A* / Dijkstra / BFS

In [ ]:
from src.path_planning import astar, dijkstra, bfs

obs_map = env.get_obstacle_map()
start   = tuple(env.agent_pos)
goal    = env.goal_pos

results = {}
for name, fn in [('A*', astar), ('Dijkstra', dijkstra), ('BFS', bfs)]:
    path, stats = fn(obs_map, start, goal)
    results[name] = (path, stats)
    status = f'{len(path)} steps, {stats["nodes_explored"]} nodes' if path else 'NO PATH'
    print(f'{name:10}: {status}')

# Plot A* path
astar_path = results['A*'][0]
fig, ax = plt.subplots(figsize=(7,7))
ax.imshow(env.render(), cmap=cmap, vmin=0, vmax=5, interpolation='nearest')
if astar_path:
    pr = [p[0] for p in astar_path]
    pc = [p[1] for p in astar_path]
    ax.plot(pc, pr, color='red', lw=2.5, label='A* path')
ax.set_title('A* Planned Path', fontsize=14)
ax.legend()
plt.tight_layout(); plt.show()

## 3. Perception — LiDAR Scan

In [ ]:
from src.perception import PerceptionSystem

perc    = PerceptionSystem(lidar_range=8)
percept = perc.perceive(obs_map, list(env.agent_pos))

print('LiDAR:', percept['lidar'])
print(f'Detected {len(percept["detections"])} objects')

# Polar plot
dirs  = ['N','NE','E','SE','S','SW','W','NW']
angles= np.linspace(0, 2*np.pi, 9)[:-1]
dists = [percept['lidar'].get(d, 8) for d in dirs]
ap    = np.append(angles, angles[0])
dp    = np.append(dists,  dists[0])

fig, ax = plt.subplots(figsize=(5,5), subplot_kw={'projection':'polar'})
ax.plot(ap, dp, color='#0077B6', lw=2)
ax.fill(ap, dp, color='#ADE8F4', alpha=0.4)
ax.set_xticks(angles); ax.set_xticklabels(dirs)
ax.set_title('LiDAR Scan', pad=20)
plt.tight_layout(); plt.show()

## 4. Full Navigation Run

In [ ]:
from src.navigation import NavigationController

ctrl    = NavigationController(algorithm='astar', replan_interval=8, max_steps=400)
summary = ctrl.run(env, verbose=True)

fig, ax = plt.subplots(figsize=(7,7))
ax.imshow(env.render(), cmap=cmap, vmin=0, vmax=5, interpolation='nearest')
status = 'SUCCESS ✓' if summary['success'] else 'FAILED ✗'
ax.set_title(f'Navigation Result — {status}  ({summary["steps"]} steps)', fontsize=13)
plt.tight_layout(); plt.show()

## 5. Algorithm Comparison

In [ ]:
algos   = ['A*', 'Dijkstra', 'BFS']
nodes   = [results[a][1]['nodes_explored'] for a in algos]
lengths = [len(results[a][0]) if results[a][0] else 0 for a in algos]

x = np.arange(len(algos)); w = 0.35
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(x - w/2, nodes,   w, label='Nodes explored', color='#0077B6', alpha=0.85)
ax.bar(x + w/2, lengths, w, label='Path length',    color='#2DC653', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(algos, fontsize=12)
ax.set_ylabel('Value'); ax.legend()
ax.set_title('Algorithm Comparison: A* vs Dijkstra vs BFS', fontsize=13)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()
print('A* is fastest: explores fewer nodes due to the heuristic.')